In [25]:
import pandas as pd
import os

daily_file = "processed_stck_data/CMG_RRV.csv"
df_daily = pd.read_csv(daily_file, parse_dates=["Date"])

In [26]:
start_date = pd.to_datetime("2018-03-31")
end_date   = pd.to_datetime("2024-12-31")

In [27]:
# Filter daily data to this date range
df_daily = df_daily[(df_daily["Date"] >= start_date) & (df_daily["Date"] <= end_date)]
df_daily.sort_values("Date", inplace=True)

In [28]:
df_daily.head()

,Date,Close,Open,High,Low,Volume,Change,Symbol,return_day,return_week,...,MFI14,MOM1,MOM3,MOM7,CCI12,CCI20,ROCR3,ROCR12,WILLR,TRIX
89,2018-03-31,20865.1,20259.533333,21162.4,19951.233333,321393.333333,3.50,CMG_RRV,-0.000527,0.165433,...,80.362711,-11.0,1904.8,2609.466667,161.353392,171.387334,1.100463,1.180680,-7.407496,0.352924
90,2018-04-01,20854.1,20699.966667,21151.4,20083.366667,246696.666667,1.67,CMG_RRV,-0.000527,0.127377,...,80.721342,-11.0,1035.0,2774.633333,123.281852,155.602049,1.052222,1.103730,-8.123028,0.417055
91,2018-04-02,20843.1,21140.400000,21140.4,20215.500000,172000.000000,-0.16,CMG_RRV,-0.000527,0.099302,...,80.533241,-11.0,-33.0,2939.800000,98.664781,136.761751,0.998419,1.091696,-9.433639,0.477710
92,2018-04-03,20479.8,20545.800000,20810.1,20413.700000,165830.000000,-1.74,CMG_RRV,-0.017430,0.033337,...,73.491633,-363.3,-385.3,1981.900000,72.662642,111.191595,0.981534,1.087719,-20.790120,0.528100
93,2018-04-04,20744.0,20479.800000,21140.4,20479.800000,288310.000000,1.29,CMG_RRV,0.012901,-0.006328,...,73.651033,264.2,-110.1,1783.700000,73.464667,109.463401,0.994720,1.125446,-12.870931,0.571061


In [29]:
df_daily.tail()

,Date,Close,Open,High,Low,Volume,Change,Symbol,return_day,return_week,...,MFI14,MOM1,MOM3,MOM7,CCI12,CCI20,ROCR3,ROCR12,WILLR,TRIX
2552,2024-12-27,48559.000000,49730.000000,49910.000000,48468.000000,3.600000e+06,-2.710000,CMG_RRV,-0.027069,-0.038061,...,52.644118,-1351.000000,-991.000000,-2342.000000,-179.602895,-140.797837,0.980000,0.976459,-97.114775,0.013771
2553,2024-12-28,48739.333333,49486.666667,49773.333333,48645.333333,2.721233e+06,-1.436667,CMG_RRV,0.003714,-0.030449,...,52.228416,180.333333,-1620.666667,-1951.333333,-139.486440,-121.079448,0.967818,0.971290,-91.397168,-0.000736
2554,2024-12-29,48919.666667,49243.333333,49636.666667,48822.666667,1.842467e+06,-0.163333,CMG_RRV,0.003700,-0.012721,...,51.192752,180.333333,-990.333333,-1560.666667,-110.536554,-103.269138,0.980158,0.987279,-85.679560,-0.015550
2555,2024-12-30,49100.000000,49000.000000,49500.000000,49000.000000,9.637000e+05,1.110000,CMG_RRV,0.003686,-0.025020,...,49.455520,180.333333,541.000000,-1170.000000,-83.565522,-86.823807,1.011141,0.980236,-79.961953,-0.029205
2556,2024-12-31,49250.000000,49300.000000,49850.000000,48950.000000,9.573000e+05,0.310000,CMG_RRV,0.003055,-0.013224,...,50.979312,150.000000,510.666667,-300.000000,-57.528557,-65.405366,1.010478,0.967564,-75.206088,-0.040865


In [30]:
df_daily.isna().sum()

Date      0
Close     0
Open      0
High      0
Low       0
         ..
CCI20     0
ROCR3     0
ROCR12    0
WILLR     0
TRIX      0
Length: 62, dtype: int64

In [31]:
# ======== Step 2: Define a function to load & process quarterly data ========
quarter_end_map = {1: '-03-31', 2: '-06-30', 3: '-09-30', 4: '-12-31'}

def load_and_process_quarterly(file_path):
    df = pd.read_csv(file_path)
    if not {"Năm", "Kỳ"}.issubset(df.columns):
        raise ValueError(f"File {file_path} does not contain 'Năm' and 'Kỳ' columns.")
    df["Quarter_End"] = pd.to_datetime(df["Năm"].astype(str) + df["Kỳ"].map(quarter_end_map))
    df = df[(df["Quarter_End"] >= start_date) & (df["Quarter_End"] <= end_date)]
    # Sort by Quarter_End
    df.sort_values("Quarter_End", inplace=True)
    return df

In [32]:
# ======== Step 3: Load each quarterly CSV file ========
cash_flow_file = "files/CMG_2025_cash_flow.csv"
financial_reports_file = "files/CMG_2025_income_statement.csv"
stock_ratio_file = "files/CMG_2025_ratio.csv"

df_cash_flow = load_and_process_quarterly(cash_flow_file)
df_financial  = load_and_process_quarterly(financial_reports_file)
df_stock_ratio = load_and_process_quarterly(stock_ratio_file)

In [33]:
df_merged = pd.merge_asof(
    df_daily.sort_values("Date"),
    df_cash_flow,
    left_on="Date",
    right_on="Quarter_End",
    direction="backward",
    suffixes=("", "_cf")
)

df_merged = pd.merge_asof(
    df_merged.sort_values("Date"),
    df_financial,
    left_on="Date",
    right_on="Quarter_End",
    direction="backward",
    suffixes=("", "_fr")
)

df_merged = pd.merge_asof(
    df_merged.sort_values("Date"),
    df_stock_ratio,
    left_on="Date",
    right_on="Quarter_End",
    direction="backward",
    suffixes=("", "_sr")
)

In [16]:
df_merged.columns.tolist()

['Date',
 'Close',
 'Open',
 'High',
 'Low',
 'Volume',
 'Change',
 'Symbol',
 'return_day',
 'return_week',
 'return_month',
 'volatility_day',
 'volatility_week',
 'volatility_month',
 'liquidity_day',
 'liquidity_week',
 'liquidity_month',
 'high_minus_close',
 'low_minus_open',
 'cumulative_return',
 'Stochastic_Osc',
 'ATR',
 'ADX14',
 'ADX20',
 'SMA_3',
 'SMA_7',
 'SMA_14',
 'SMA_21',
 'SMA_50',
 'SMA_100',
 'WMA_3',
 'WMA_7',
 'WMA_14',
 'WMA_21',
 'WMA_50',
 'WMA_100',
 'EMA6',
 'EMA12',
 'EMA26',
 'outMACD',
 'outMACDSignal',
 'outMACDHist',
 'RSI6',
 'RSI12',
 'RSI14',
 'StochRSI_6',
 'StochRSI_12',
 'StochRSI_14',
 'BBANDSMIDDLE',
 'BBANDSUPPER',
 'BBANDSLOWER',
 'OBV',
 'MFI14',
 'MOM1',
 'MOM3',
 'MOM7',
 'CCI12',
 'CCI20',
 'ROCR3',
 'ROCR12',
 'WILLR',
 'TRIX',
 'CP',
 'Năm',
 'Kỳ',
 'Lãi/Lỗ ròng trước thuế',
 'Khấu hao TSCĐ',
 'Dự phòng RR tín dụng',
 'Lãi/Lỗ chênh lệch tỷ giá chưa thực hiện',
 'Lãi/Lỗ từ hoạt động đầu tư',
 'Thu nhập lãi',
 'Thu lãi và cổ tức',
 'Lưu c

In [34]:
cols_to_drop = ["CP_sr", "Năm_sr", "Kỳ_sr", "Quarter_End_sr", "Quarter_End_fr", "Quarter_End", "CP_fr", "Năm_fr", "Kỳ_fr", "CP", "Năm", "Kỳ"]
df_merged.drop(columns=cols_to_drop, inplace=True)

In [35]:
rename_dict = {
    'Date': 'date',
    'Close': 'close',
    'Open': 'open',
    'High': 'high',
    'Low': 'low',
    'Volume': 'volume',
    'Change': 'change',
    'Symbol': 'symbol',
    'return_day': 'return_day',
    'return_week': 'return_week',
    'return_month': 'return_month',
    'volatility_day': 'volatility_day',
    'volatility_week': 'volatility_week',
    'volatility_month': 'volatility_month',
    'liquidity_day': 'liquidity_day',
    'liquidity_week': 'liquidity_week',
    'liquidity_month': 'liquidity_month',
    'high_minus_close': 'high_minus_close',
    'low_minus_open': 'low_minus_open',
    'cumulative_return': 'cumulative_return',
    'Stochastic_Osc': 'stochastic_osc',
    'ATR': 'atr',
    'ADX14': 'adx_14',
    'ADX20': 'adx_20',
    'SMA_3': 'sma_3',
    'SMA_7': 'sma_7',
    'SMA_14': 'sma_14',
    'SMA_21': 'sma_21',
    'SMA_50': 'sma_50',
    'SMA_100': 'sma_100',
    'WMA_3': 'wma_3',
    'WMA_7': 'wma_7',
    'WMA_14': 'wma_14',
    'WMA_21': 'wma_21',
    'WMA_50': 'wma_50',
    'WMA_100': 'wma_100',
    'EMA6': 'ema_6',
    'EMA12': 'ema_12',
    'EMA26': 'ema_26',
    'outMACD': 'out_macd',
    'outMACDSignal': 'out_macd_signal',
    'outMACDHist': 'out_macd_hist',
    'RSI6': 'rsi_6',
    'RSI12': 'rsi_12',
    'RSI14': 'rsi_14',
    'StochRSI_6': 'stochrsi_6',
    'StochRSI_12': 'stochrsi_12',
    'StochRSI_14': 'stochrsi_14',
    'BBANDSMIDDLE': 'bbands_middle',
    'BBANDSUPPER': 'bbands_upper',
    'BBANDSLOWER': 'bbands_lower',
    'OBV': 'obv',
    'MFI14': 'mfi_14',
    'MOM1': 'mom_1',
    'MOM3': 'mom_3',
    'MOM7': 'mom_7',
    'CCI12': 'cci_12',
    'CCI20': 'cci_20',
    'ROCR3': 'rocr_3',
    'ROCR12': 'rocr_12',
    'WILLR': 'willr',
    'TRIX': 'trix',
    'Lãi/Lỗ ròng trước thuế': 'net_profit_loss_before_tax',
    'Khấu hao TSCĐ': 'depreciation_fixed_assets',
    'Dự phòng RR tín dụng': 'credit_risk_reserve',
    'Lãi/Lỗ chênh lệch tỷ giá chưa thực hiện': 'unrealized_forex_gain_loss',
    'Lãi/Lỗ từ hoạt động đầu tư': 'investment_income_loss',
    'Thu nhập lãi': 'interest_income',
    'Thu lãi và cổ tức': 'interest_dividend_income',
    'Lưu chuyển tiền thuần từ HĐKD trước thay đổi VLĐ': 'net_cash_flow_operating_before_wc_changes',
    'Tăng/Giảm các khoản phải thu': 'change_in_receivables',
    'Tăng/Giảm hàng tồn kho': 'change_in_inventories',
    'Tăng/Giảm các khoản phải trả': 'change_in_payables',
    'Tăng/Giảm chi phí trả trước': 'change_in_prepaid_expenses',
    'Chi phí lãi vay đã trả': 'interest_expense_paid',
    'Tiền thu nhập doanh nghiệp đã trả': 'corporate_income_tax_paid',
    'Tiền thu khác từ các hoạt động kinh doanh': 'other_cash_inflows_operating',
    'Tiền chi khác từ các hoạt động kinh doanh': 'other_cash_outflows_operating',
    'Lưu chuyển tiền tệ ròng từ các hoạt động SXKD': 'net_cash_flow_from_business_activities',
    'Mua sắm TSCĐ': 'capital_expenditure',
    'Tiền thu được từ thanh lý tài sản cố định': 'cash_from_fixed_assets_sale',
    'Tiền chi cho vay, mua công cụ nợ của đơn vị khác (đồng)': 'cash_outflow_loans_debt_instruments',
    'Tiền thu hồi cho vay, bán lại các công cụ nợ của đơn vị khác (đồng)': 'cash_inflow_loan_recoveries',
    'Đầu tư vào các doanh nghiệp khác': 'investments_in_other_businesses',
    'Tiền thu từ việc bán các khoản đầu tư vào doanh nghiệp khác': 'cash_from_sale_investments',
    'Tiền thu cổ tức và lợi nhuận được chia': 'dividends_profit_received',
    'Lưu chuyển từ hoạt động đầu tư': 'cash_flow_investing',
    'Tăng vốn cổ phần từ góp vốn và/hoặc phát hành cổ phiếu': 'increase_in_equity',
    'Chi trả cho việc mua lại, trả cổ phiếu': 'cash_used_share_repurchase',
    'Tiền thu được các khoản đi vay': 'cash_from_borrowings',
    'Tiền trả các khoản đi vay': 'cash_repayment_borrowings',
    'Cổ tức đã trả': 'dividends_paid',
    'Lưu chuyển tiền từ hoạt động tài chính': 'cash_flow_financing',
    'Lưu chuyển tiền thuần trong kỳ': 'net_cash_flow_period',
    'Tiền và tương đương tiền': 'cash_and_equivalents',
    'Ảnh hưởng của chênh lệch tỷ giá': 'exchange_rate_effect',
    'Tiền và tương đương tiền cuối kỳ': 'ending_cash_equivalents',
    'Tăng trưởng doanh thu (%)': 'revenue_growth_percent',
    'Doanh thu (đồng)': 'revenue_vnd',
    'Lợi nhuận sau thuế của Cổ đông công ty mẹ (đồng)': 'net_profit_after_tax_parent_vnd',
    'Tăng trưởng lợi nhuận (%)': 'profit_growth_percent',
    'Thu nhập tài chính': 'financial_income',
    'Chi phí tiền lãi vay': 'interest_expense',
    'Doanh thu bán hàng và cung cấp dịch vụ': 'sales_revenue_service_income',
    'Các khoản giảm trừ doanh thu': 'revenue_deductions',
    'Doanh thu thuần': 'net_revenue',
    'Giá vốn hàng bán': 'cogs',
    'Lãi gộp': 'gross_profit',
    'Chi phí tài chính': 'financial_expenses',
    'Lãi/lỗ từ công ty liên doanh': 'profit_loss_associates',
    'Chi phí bán hàng': 'selling_expenses',
    'Chi phí quản lý DN': 'administrative_expenses',
    'Lãi/Lỗ từ hoạt động kinh doanh': 'operating_profit_loss',
    'Thu nhập khác': 'other_income',
    'Lãi lỗ trong công ty liên doanh, liên kết': 'profit_loss_joint_ventures',
    'Thu nhập/Chi phí khác': 'other_income_expense',
    'Lợi nhuận khác': 'other_profit',
    'LN trước thuế': 'profit_before_tax',
    'Chi phí thuế TNDN hiện hành': 'current_corporate_tax_expense',
    'Chi phí thuế TNDN hoãn lại': 'deferred_corporate_tax_expense',
    'Lợi nhuận thuần': 'net_profit',
    'Cổ đông thiểu số': 'minority_interest',
    'Cổ đông của Công ty mẹ': 'parent_company_shareholders',
    '(Vay NH+DH)/VCSH': 'loans_to_equity_ratio',
    'Nợ/VCSH': 'debt_to_equity',
    'TSCĐ / Vốn CSH': 'fixed_assets_to_equity',
    'Vốn CSH/Vốn điều lệ': 'equity_to_chartered_capital',
    'Vòng quay tài sản': 'asset_turnover',
    'Vòng quay TSCĐ': 'fixed_asset_turnover',
    'Số ngày thu tiền bình quân': 'avg_collection_period',
    'Số ngày tồn kho bình quân': 'avg_inventory_days',
    'Số ngày thanh toán bình quân': 'avg_payment_days',
    'Chu kỳ tiền': 'cash_cycle',
    'Vòng quay hàng tồn kho': 'inventory_turnover',
    'Biên EBIT (%)': 'ebit_margin_percent',
    'Biên lợi nhuận gộp (%)': 'gross_profit_margin_percent',
    'Biên lợi nhuận ròng (%)': 'net_profit_margin_percent',
    'ROE (%)': 'roe_percent',
    'ROIC (%)': 'roic_percent',
    'ROA (%)': 'roa_percent',
    'EBITDA (Tỷ đồng)': 'ebitda_billion_vnd',
    'EBIT (Tỷ đồng)': 'ebit_billion_vnd',
    'Tỷ suất cổ tức (%)': 'dividend_yield_percent',
    'Chỉ số thanh toán hiện thời': 'current_ratio',
    'Chỉ số thanh toán tiền mặt': 'cash_ratio',
    'Chỉ số thanh toán nhanh': 'quick_ratio',
    'Khả năng chi trả lãi vay': 'interest_coverage',
    'Đòn bẩy tài chính': 'financial_leverage',
    'Vốn hóa (Tỷ đồng)': 'market_cap_billion_vnd',
    'Số CP lưu hành (Triệu CP)': 'shares_outstanding_million',
    'P/E': 'pe',
    'P/B': 'pb',
    'P/S': 'ps',
    'P/Cash Flow': 'p_cash_flow',
    'EPS (VND)': 'eps_vnd',
    'BVPS (VND)': 'bvps_vnd',
    'EV/EBITDA': 'ev_ebitda'
}

df_merged = df_merged.rename(columns=rename_dict)

In [36]:
# ======== Step 5: Save the combined DataFrame ========
output_file = "feature_engineered/feature_engineered_CMG_RRV.csv"
df_merged.to_csv(output_file, index=False)
print(f"Saved combined feature-engineered file to {output_file}")

Saved combined feature-engineered file to feature_engineered/feature_engineered_CMG_RRV.csv


# The End